# Fine-tuning DistilBERT for NER on Tweet Data

**Input:** `tweets_cleaned.csv` (`id`, `entities`, `clean_text`)
**Model:** `distilbert-base-cased`
**Task:** Token classification (NER) - Beginning, Intermediate, Outside (BIO) tagging for PER/LOC/ORG/MISC

**Note on compute:** this notebook is written for **CPU training**.
`distilbert-base-cased` has ~66M parameters (about 40% fewer than
`bert-base-cased`'s ~110M, via knowledge distillation - it's trained to
mimic BERT's outputs with 6 transformer layers instead of 12), so it
trains noticeably faster on CPU while typically losing only a small
amount of accuracy - a reasonable trade for an NER task this size. The
notebook still prints a time estimate before the real training run so
we know what to expect on your machine before committing to it. If
we do have any GPU available, just running this same code there will
pick it up automatically - no changes needed.

## 0. Install dependencies

In [2]:
# !pip install torch transformers datasets seqeval scikit-learn nltk accelerate -q

## 1. Imports and setup

In [7]:
import json
import time
import numpy as np
import pandas as pd
import torch
from nltk.tokenize import TweetTokenizer
import nltk

nltk.download("punkt", quiet=True)
tk = TweetTokenizer()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## 2. Load cleaned data and parse entity annotations

Same parsing/alignment logic validated in the cleaning notebook -
included here so this notebook is self-contained and can run on its
own. `entities` can be empty (NaN) for tweets with no tagged entities -
those are valid all-`O` rows, not errors.

In [8]:
INPUT_PATH = "tweets_cleaned.csv"  # update path if needed

df = pd.read_csv(INPUT_PATH)
df["entities"] = df["entities"].fillna("")
print(f"Loaded {len(df)} rows")
df.head(20)

Loaded 2730 rows


,id,entities,clean_text
0,1,LOC/Thai Buddhist temple;,"2,000 fetuses found hidden at Thai Buddhist te..."
1,2,LOC/canada;,"870, 000 people in canada depend on -25% incre..."
2,3,PER/louis;,"7961212234, phone this girl! she is like louis..."
3,4,ORG/WikiLeaks;LOC/Southern Ocean;,WikiLeaks Set To Reveal US-UFO War In Southern...
4,5,ORG/queen;,"queen, bohemian rhapsody please"
5,6,PER/cheryl cole;PER/Danni;PER/eddie;,cheryl cole is starting to lose that connectio...
6,7,ORG/Lester;MISC/Mason & Begg Limited;,"Lester, "" Mason & Begg Limited "" "" 'Forbidden ..."
7,8,PER/Thomas Watson;,"To be successful, you have to have your heart ..."
8,9,"PER/Sanchez;PER/Eli Wallach;MISC/The Good, The...","Sanchez looks like Eli Wallach in The Good, Th..."
9,10,,"the history of the mayor, the city ."


In [9]:
def parse_entities(entity_str):
    """Parses a raw annotation string from a tweet into a list of structured entity tuples.

    This function cleans and extracts labeled entities formatted as 'LABEL/text;'.
    It handles malformed chunks, trailing semicolons, and enforces consistent 
    uppercase formatting for the entity labels to ensure safe downstream mapping.

    Args:
        entity_str (str): The raw annotation string from the dataset 
                          (e.g., "LOC/Netherlands; PER/Christian Bale;").

    Returns:
        list of tuple (str, str): A list of parsed entities where each element is 
                                  a tuple of (entity_label, surface_text).
                                  Example: [('LOC', 'Netherlands'), ('PER', 'Christian Bale')]
    """
    entities = []
    if not entity_str:
        return entities
    for chunk in entity_str.split(";"):
        chunk = chunk.strip()
        if not chunk or "/" not in chunk:
            continue
        label, surface = chunk.split("/", 1)
        entities.append((label.strip().upper(), surface.strip()))
    return entities


PLACEHOLDER_TOKENS = {"@FakeUsername", "http://FakeURL", "#FakeHashtag"}


def _strip_possessive(token):
    for suffix in ("'s", "\u2019s", "'", "\u2019"):
        if token.endswith(suffix):
            return token[: -len(suffix)]
    return token


def _hyphen_parts(token):
    """Splits a token by its hyphens, or wraps it in a list if unhyphenated.

    This helper is crucial for matching sub-components of hyphenated words 
    when the tweet text uses a hyphen but the entity annotation does not 
    (e.g., matching a tweet's "Spider-Man" against the target string "Man").

    Args:
        token (str): The token to inspect.

    Returns:
        list of str: A list containing the split parts of the hyphenated token,
                     or a single-element list of the original token.
    """
    return token.split("-") if "-" in token else [token]


def tag_tokens(raw_text, entity_str, verbose=False):
    """Tokenizes a raw tweet and accurately aligns its text tokens with 

    extracted entity targets into a parallel BIO tag sequence.

    This method addresses data mismatches common in user-generated text 
    by executing strict case-sensitive and relaxed case-insensitive matching,
    while systematically handling possessives, hyphens, and placeholders.

    Args:
        raw_text (str): The raw string contents of the tweet.
        entity_str (str): The raw multi-entity annotation string (e.g. "PER/Bale;").
        verbose (bool, optional): If True, logs alignment warnings for missing spans. 
                                  Defaults to False.

    Returns:
        tuple (list of str, list of str): A parallel tuple of (final_tokens, final_tags)
                                          where tags follow the BIO format.
        (["Christian", "Bale", "thinks", "Batman", "is", "awesome"], ["B-PER", "I-PER", "O", "O", "O", "O"])
    """
    tokens = tk.tokenize(raw_text)
    tags = ["O"] * len(tokens)
    consumed = [False] * len(tokens)
    entities = parse_entities(entity_str)

    for label, surface in entities:
        ent_tokens = tk.tokenize(surface)
        if not ent_tokens:
            continue
        n = len(ent_tokens)
        match_start = None

        def window_matches(window, ent, case_sensitive):
            if not case_sensitive:
                window = [w.lower() for w in window]
                ent = [e.lower() for e in ent]
            if len(ent) > 2 and window[1:-1] != ent[1:-1]:
                return False
            first_w, first_e = window[0], ent[0]
            last_w, last_e = window[-1], ent[-1]

            def first_ok(w, e):
                if w == e:
                    return True
                return _hyphen_parts(w)[-1] == e

            def last_ok(w, e):
                if w == e:
                    return True
                if _strip_possessive(w) == e:
                    return True
                return _hyphen_parts(w)[0] == e

            if len(ent) == 1:
                w = first_w
                if w == first_e:
                    return True
                if _strip_possessive(w) == first_e:
                    return True
                return first_e in _hyphen_parts(w)

            return first_ok(first_w, first_e) and last_ok(last_w, last_e)

        for i in range(len(tokens) - n + 1):
            if any(consumed[i:i + n]):
                continue
            if window_matches(tokens[i:i + n], ent_tokens, case_sensitive=True):
                match_start = i
                break
        if match_start is None:
            for i in range(len(tokens) - n + 1):
                if any(consumed[i:i + n]):
                    continue
                if window_matches(tokens[i:i + n], ent_tokens, case_sensitive=False):
                    match_start = i
                    break

        if match_start is None:
            if verbose:
                print(f"  [WARN] could not align entity '{surface}' ({label}) in: {raw_text[:80]}")
            continue

        for j in range(match_start, match_start + n):
            tags[j] = ("B-" if j == match_start else "I-") + label
            consumed[j] = True

    final_tokens, final_tags = [], []
    for t, tag in zip(tokens, tags):
        if t in PLACEHOLDER_TOKENS:
            continue
        final_tokens.append(t)
        final_tags.append(tag)

    return final_tokens, final_tags

## 3. Tag all rows and build the label set

Drops rows with zero tokens after cleaning (rare - only tweets that were
pure noise). ~0.3% of entity annotations fail to align due to genuine
typos/truncations in the source annotation data (e.g. `Murkowsk` for
`Murkowski`) - these print as warnings; the rest of each affected row's
tokens/entities are unaffected.

In [10]:
records = []
for _, row in df.iterrows():
    toks, tags = tag_tokens(row["clean_text"], row["entities"], verbose=True)
    records.append({"id": row["id"], "tokens": toks, "tags": tags})

tagged_df = pd.DataFrame(records)
tagged_df = tagged_df[tagged_df["tokens"].map(len) > 0].reset_index(drop=True)
print(f"\nTagged rows: {len(tagged_df)}")

all_labels = sorted({t for tags in tagged_df["tags"] for t in tags})
label2id = {l: i for i, l in enumerate(all_labels)}
id2label = {i: l for l, i in label2id.items()}
print("Labels:", all_labels)

tagged_df["tag_ids"] = tagged_df["tags"].apply(lambda tags: [label2id[t] for t in tags])

  [WARN] could not align entity 'Ariz.' (LOC) in: Final vote count Ariz. ... voters OK ... measure legalizing medical marijuana
  [WARN] could not align entity 'Kamal `Nath' (PER) in: GoI not taken 'action agsnt Pop killrs Mins/MP Kamal 'Nath, Bhuria, Mutemwar, SS
  [WARN] could not align entity 'U.S.' (LOC) in: Leaked documents lift curtain on U.S. foreign policy Tens of thousands of confid
  [WARN] could not align entity 'b `mouth' (LOC) in: Leigh got to meet them after their Bournemouth gig! But we had London tickets & 
  [WARN] could not align entity 'Disney' (ORG) in: wow lol! I guess that's the Disney version of the song . Just like Miley Cyrus i
  [WARN] could not align entity 'Transformers : The Movie' (MISC) in: FACT Orson Welles' last film role was as the voice actor for the character "Unic
  [WARN] could not align entity 'Amber Buffet & Hibach' (LOC) in: Hungry!! (@ Amber Buffet & Hibachi)
  [WARN] could not align entity 'Grover Washington Jr.' (PER) in: JUST THE TWO OF US??

In [11]:
tagged_df

,id,tokens,tags,tag_ids
0,1,"[2,000, fetuses, found, hidden, at, Thai, Budd...","[O, O, O, O, O, B-LOC, I-LOC, I-LOC, O]","[8, 8, 8, 8, 8, 0, 4, 4, 8]"
1,2,"[870, ,, 000, people, in, canada, depend, on, ...","[O, O, O, O, O, B-LOC, O, O, O, O, O, O, O, O,...","[8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, ..."
2,3,"[7961212234, ,, phone, this, girl, !, she, is,...","[O, O, O, O, O, O, O, O, O, B-PER, O, O, O, O,...","[8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, ..."
3,4,"[WikiLeaks, Set, To, Reveal, US-UFO, War, In, ...","[B-ORG, O, O, O, O, O, O, B-LOC, I-LOC, O, O, ...","[2, 8, 8, 8, 8, 8, 8, 0, 4, 8, 8, 8, 8, 8, 8, ..."
4,5,"[queen, ,, bohemian, rhapsody, please]","[B-ORG, O, O, O, O]","[2, 8, 8, 8, 8]"
...,...,...,...,...
2725,2811,"[RT, On, this, date, in, 1969, As, a, protest,...","[O, O, O, O, O, O, O, O, O, O, B-LOC, O, O, O,...","[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, ..."
2726,2812,"[RT, A, good, friend, understands, you, even, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]"
2727,2813,"[RT, A, pop-up, experience, celebrating, Micha...","[O, O, O, O, O, B-PER, I-PER, O, O, O, O, O]","[8, 8, 8, 8, 8, 3, 7, 8, 8, 8, 8, 8]"
2728,2814,"[RT, Hugh, Hefner, is, 84, and, engaged, to, a...","[O, B-PER, I-PER, O, O, O, O, O, O, O, O, O, O...","[8, 3, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ..."


## 4. Train / validation / test split

80/10/10 split. `random_state` fixed for reproducibility.

In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(tagged_df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

train=2184  val=273  test=273


## 5. Load tokenizer and align subword tokens to labels

BERT splits words into subword pieces (`WikiLeaks` -> `W`, `##ik`,
`##ile`, `##aks`). Our labels are per-**word**, not per-subword, so each
word's label must be copied/ignored across its subwords:

- First subword of a word -> gets that word's real label
- Continuation subwords -> label `-100`, which `CrossEntropyLoss`
  ignores by default, so they don't contribute to training loss
- Special tokens (`[CLS]`, `[SEP]`, padding) -> also `-100`

This is the standard Hugging Face token-classification recipe, using
`tokenized.word_ids()` to know which original word each subword token
came from.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-cased"
MAX_LENGTH = 64  # tweets are short; generous upper bound

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_and_align(examples):
    """Tokenizes pre-split word sequences and aligns their corresponding word-level 

    BIO labels with the newly generated subword (WordPiece) token IDs.

    This function maps a single word-level label to only the first subword 
    segment of a split word, masking all subsequent subword pieces, special 
    tokens ([CLS], [SEP]), and padding positions with a loss-ignored index (-100).

    Args:
        examples (dict): A batch of data from a Hugging Face Dataset containing:
            - "tokens" (list of list of str): Pre-tokenized words per tweet.
            - "tag_ids" (list of list of int): Mapped numeric BIO IDs per word.

    Returns:
        dict: 'input_ids', 'attention_mask', 'labels'
    """
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    all_labels = []
    for i, tag_ids in enumerate(examples["tag_ids"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev_word_id:
                label_ids.append(tag_ids[wid])
            else:
                label_ids.append(-100)
            prev_word_id = wid
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

/Users/aniketchoudhary/miniconda3/envs/dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 6. Build Hugging Face `Dataset` objects

In [8]:
from datasets import Dataset, DatasetDict

train_ds = Dataset.from_pandas(train_df[["tokens", "tag_ids"]].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[["tokens", "tag_ids"]].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[["tokens", "tag_ids"]].reset_index(drop=True))

raw_datasets = DatasetDict({"train": train_ds, "validation": val_ds, "test": test_ds})

tokenized_datasets = raw_datasets.map(
    tokenize_and_align,
    batched=True,
    remove_columns=["tokens", "tag_ids"],
)
tokenized_datasets

Map: 100%|██████████| 273/273 [00:00<00:00, 11404.04 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2184
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 273
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 273
    })
})

## 7. Load the pretrained model

This downloads `distilbert-base-cased` weights from the Hugging Face
Hub - requires internet access. `num_labels` and the `id2label`/
`label2id` mappings are passed so the model's output layer matches our
9-label BIO tag set and predictions print as readable labels later.

In [9]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(all_labels),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)
print(f"Model loaded on {device}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11339.02it/s]
[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cpu


## 8. Metrics - `seqeval`

NER evaluation should be **entity-level**, not token-level: a
multi-token entity like `B-LOC I-LOC` only counts as correct if the
*entire span* is predicted correctly, not each token independently.
`seqeval` handles this properly and reports per-entity-type precision/
recall/F1 plus overall scores - this is the standard NER metric

In [10]:
from seqeval.metrics import precision_score, recall_score, f1_score, accuracy_score


def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [id2label[l] for l in label_row if l != -100]
        for label_row in labels
    ]
    true_predictions = [
        [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": accuracy_score(true_labels, true_predictions),
    }

## 9. Training configuration (CPU-aware)

Settings chosen for feasibility on CPU:
- Small batch size (8) - large batches are mainly a GPU-memory
  optimization; on CPU they mostly just mean slower per-step wall time
  with no real speed benefit, so a moderate batch keeps memory low
  without extra cost
- 3 epochs - enough for a 110M-parameter model to adapt to a small
  (~2,200 example) dataset without taking excessively long
- `fp16=False` - mixed-precision training requires a GPU; CPU training
  uses standard fp32

**Before running the real `trainer.train()` call below, this notebook
times a single training step on the actual machine and extrapolates a
rough total-time estimate**, so we know what we are committing to
before launching a multi-epoch run.

In [11]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./ner_distilbert_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

### 9a. Time estimate before committing to the full run

In [12]:
# time a few training steps to estimate total run time
import copy

probe_args = copy.deepcopy(training_args)
probe_args.num_train_epochs = 1
probe_args.max_steps = 5
probe_args.eval_strategy = "no"
probe_args.save_strategy = "no"
probe_args.report_to = "none"

probe_trainer = Trainer(
    model=model,
    args=probe_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)

start = time.time()
probe_trainer.train()
elapsed = time.time() - start
sec_per_step = elapsed / 5

steps_per_epoch = len(tokenized_datasets["train"]) // training_args.per_device_train_batch_size
total_steps = steps_per_epoch * training_args.num_train_epochs
est_total_minutes = (sec_per_step * total_steps) / 60

print(f"~{sec_per_step:.2f} sec/step on this machine")
print(f"Estimated total training time for {training_args.num_train_epochs} epochs: ~{est_total_minutes:.1f} minutes")
print("(Re-run cell 9 above to get a fresh, un-trained model before the real training run below,")
print(" since this probe already performed 5 optimizer steps on it.)")

/Users/aniketchoudhary/miniconda3/envs/dev/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


~1.69 sec/step on this machine
Estimated total training time for 3 epochs: ~23.1 minutes
(Re-run cell 9 above to get a fresh, un-trained model before the real training run below,
 since this probe already performed 5 optimizer steps on it.)


## 10. Train

In [13]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.205014,0.179702,0.670455,0.606164,0.636691,0.949451
2,0.096673,0.153382,0.718861,0.691781,0.705061,0.957802
3,0.079613,0.156366,0.714286,0.719178,0.716724,0.959780


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.91it/s]
/Users/aniketchoudhary/miniconda3/envs/dev/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.15it/s]
/Users/aniketchoudhary/miniconda3/envs/dev/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.12it/s]


## 11. Evaluate on the held-out test set

In [14]:
test_results = trainer.evaluate(tokenized_datasets["test"])
test_results

/Users/aniketchoudhary/miniconda3/envs/dev/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.079613,0.180834,3,0.664516,0.693603,0.678748,0.948897


{'eval_loss': 0.1808335781097412,
 'eval_precision': 0.6645161290322581,
 'eval_recall': 0.6936026936026936,
 'eval_f1': 0.6787479406919275,
 'eval_accuracy': 0.9488971391133435}

## 12. Per-entity-type breakdown

Overall F1 hides whether the model is good at all entity types equally -
PER tends to be easiest (capitalized names are a strong, learnable
signal), MISC tends to be hardest (it's a catch-all category with the
least consistent surface patterns)

In [15]:
from seqeval.metrics import classification_report

predictions, labels, _ = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions, axis=2)

true_labels = [[id2label[l] for l in row if l != -100] for row in labels]
true_predictions = [
    [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
    for pred_row, label_row in zip(predictions, labels)
]

print(classification_report(true_labels, true_predictions))

              precision    recall  f1-score   support

         LOC       0.68      0.76      0.72        58
        MISC       0.08      0.04      0.05        27
         ORG       0.56      0.57      0.56        53
         PER       0.74      0.82      0.78       159

   micro avg       0.66      0.69      0.68       297
   macro avg       0.51      0.55      0.53       297
weighted avg       0.63      0.69      0.66       297



## 13. Inspect example predictions

In [16]:
for i in range(3):
    example_tokens = test_df.iloc[i]["tokens"]
    true_tags = test_df.iloc[i]["tags"]
    pred_tags_for_row = true_predictions[i][: len(example_tokens)]
    print(f"Tweet: {' '.join(example_tokens)}")
    for tok, true_t, pred_t in zip(example_tokens, true_tags, pred_tags_for_row):
        marker = "" if true_t == pred_t else "  <-- MISMATCH"
        if true_t != "O" or pred_t != "O":
            print(f"  {tok:20s} true={true_t:10s} pred={pred_t:10s}{marker}")
    print()

Tweet: Can i have a goodnight tweet or follow pleasee ;) Love u loads & whens the next time your going to leeds or sheffield x x x
  leeds                true=B-LOC      pred=B-LOC     
  sheffield            true=B-LOC      pred=O           <-- MISMATCH

Tweet: A TrUe FaNtasTic LiFe Never There-CAKE music video

Tweet: Brandon Flowers and crossfire ! . ( 3
  Brandon              true=B-PER      pred=B-PER     
  Flowers              true=I-PER      pred=I-PER     



## 14. Save the fine-tuned model

Saves both model weights and tokenizer so the model can be reloaded
later with `AutoModelForTokenClassification.from_pretrained(SAVE_PATH)`
without needing to retrain.

In [17]:
SAVE_PATH = "./ner_distilbert_finetuned"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.34it/s]

Model saved to ./ner_distilbert_finetuned
